# EVE Timing Debug

Runs `debug_eve_timing.py` which tests four scenarios back-to-back.
All output is printed inline in the cell below.

## What each scenario measures

| Scenario | What runs | `record_diagnostics` | Purpose |
|----------|-----------|---------------------|---------|
| **A** | `timed_eve_step()` manual reimpl | — | Baseline: clean allocator, no backward |
| **B** | `timed_eve_step()` manual reimpl | — | Real allocator state (after backward) |
| **C** | **actual `opt.step()`** | `False` | Confirms real step matches Scenario B |
| **D** | **actual `opt.step()`** | `True`  | Measures `_record_step_diagnostics` cost |

## What to look for

**C ≈ B (~70ms)** — the actual `opt.step()` is as fast as the manual reimplementation. The fix is fully working end-to-end.

**D - C = `_record_step_diagnostics` overhead.** This is the extra cost per step when `record_diagnostics=True` is set in the training notebook (cell 23 of `exp1_internal_behaviour_v2.ipynb`). It includes ~269 MB of synchronous GPU→CPU transfers (direction vectors + combined update + strength signal), 62 extra `einsum` ops, and 10 CPU-side cosine-similarity calls on 11.2M-element vectors.

**`New retries` in Scenario D** — if > 0, the `_record_step_diagnostics` allocations are causing the CUDA allocator to call `cudaMalloc`, confirming allocator-fragmentation overhead.

## Expected results (L4 GPU, ResNet-18, batch=128, K=4)

| Scenario | Expected mean batch time |
|----------|--------------------------|
| A (EVE phases only) | ~53 ms |
| B (EVE phases only) | ~56 ms |
| B full batch (fwd+bwd+EVE) | ~70 ms |
| C `record_diagnostics=False` | ~70 ms |
| D `record_diagnostics=True`  | ~70 ms + diagnostics overhead |


In [ ]:
import os, sys, subprocess

BRANCH = "full-batch-probing"

if os.path.exists("/content"):
    repo_dir = "/content/EVE"
    if os.path.exists(repo_dir):
        subprocess.run(["git", "-C", repo_dir, "fetch", "origin"], check=False)
        subprocess.run(["git", "-C", repo_dir, "checkout", BRANCH], check=False)
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{BRANCH}"], check=False)
        print(f"Updated existing clone to origin/{BRANCH}")
    else:
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/EVE.git", repo_dir],
            check=False,
        )
        print(f"Cloned branch {BRANCH}")
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    result = subprocess.run(
        ["git", "-C", repo_dir, "log", "--oneline", "-3"],
        capture_output=True, text=True,
    )
    print("Active commits:\n" + result.stdout.strip())
else:
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if parent not in sys.path:
        sys.path.insert(0, parent)


In [ ]:
# Run the full debug script — all output printed inline.
script = "/content/EVE/debug_eve_timing.py" if os.path.exists("/content") else "../debug_eve_timing.py"
%run $script
